# Final Cleaning — Work-Type & Seniority Categorization

Collapses LinkedIn's raw `workType` taxonomy into broader categories and
extracts a seniority tier from job titles via regex.

**Input:** `data/mia_postings_final.csv` (output of `02b_enrich_ai_flag_language_geo.ipynb`)
**Output:** `data/mia_postings_final2.csv`

In [ ]:
import re
import pandas as pd

In [ ]:
INPUT_PATH = "data/mia_postings_final.csv"
OUTPUT_PATH = "data/mia_postings_final2.csv"

In [ ]:
df = pd.read_csv(INPUT_PATH)

In [ ]:
MARKETING_KW = ["Marketing", "Advertising", "Product Management", "Public Relations"]
ANALYTICS_KW = ["Information Technology", "Analyst", "Engineering", "Research", "Quality Assurance"]
BUSINESS_KW  = ["Business Development", "Sales", "Consulting", "Strategy/Planning",
                "Finance", "General Business", "Accounting/Auditing", "Management"]

In [ ]:
def collapse_worktype(val):
    """Collapse LinkedIn's raw workType field into one of five broad buckets.
    Postings can carry multiple workType tags (e.g. both 'Marketing' and
    'Analyst'), which is why this checks membership rather than exact match.
    """
    if pd.isna(val):
        return "Other/Unspecified"
    has_mkt = any(kw in val for kw in MARKETING_KW)
    has_ana = any(kw in val for kw in ANALYTICS_KW)
    has_biz = any(kw in val for kw in BUSINESS_KW)
    if has_mkt and has_ana:
        return "Marketing + Analytics Hybrid"
    if has_mkt:
        return "Marketing-Focused"
    if has_ana:
        return "Analytics/IT-Focused"
    if has_biz:
        return "Business/Sales-Focused"
    return "Other/Mixed"

In [ ]:
df["workType_category"] = df["workType"].apply(collapse_worktype)

In [ ]:
def extract_seniority(title):
    """Regex-based seniority tier from job title. Feeds the seniority-tier
    findings in Inferences.md (e.g. Entry/Junior being the most crowded
    AI-adoption segment, not the least).
    """
    if pd.isna(title):
        return "Mid/Unspecified"
    t = title.lower()
    if re.search(r"\b(director|vp|vice president|head of|chief)\b", t):
        return "Director/Executive"
    if re.search(r"\b(manager|lead|principal|staff)\b", t):
        return "Manager/Lead"
    if re.search(r"\b(senior|sr\.?)\b", t) or re.search(r"\b(ii|iii)\b", t):
        return "Senior"
    if re.search(r"\b(junior|jr\.?|entry level|entry-level|associate|intern|graduate|trainee)\b", t):
        return "Entry/Junior"
    return "Mid/Unspecified"

In [ ]:
df["seniority"] = df["title"].apply(extract_seniority)

In [ ]:
print("workType_category:")
print(df["workType_category"].value_counts())
print()
print("seniority:")
print(df["seniority"].value_counts())

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH} — shape: {df.shape}")